Arrays and linked lists store data in a straight line — one element after another. Trees break that constraint. A tree arranges data hierarchically: one root, branching into children, which branch into their own children. This shape maps naturally onto file systems, organisation charts, HTML documents, expression parsers, and — most importantly for algorithms — it enables search, insertion, and deletion in **O(log n)** time when the tree is balanced.

Binary trees, where every node has at most two children, are the foundation for binary search trees, heaps, tries, and segment trees. Before you can understand any of those, you need to understand how trees are structured, how to traverse them, and why recursion is the natural tool for both.

## Tree Terminology

Every tree is made of **nodes** connected by **edges**. The vocabulary you need:

- **Root** — the single node at the top. Every tree has exactly one root.
- **Parent / Child** — if node A has an edge to node B below it, A is B's parent and B is A's child.
- **Leaf** — a node with no children.
- **Internal node** — any node that is not a leaf.
- **Depth of a node** — the number of edges from the root to that node. The root has depth 0.
- **Height of a node** — the number of edges on the longest path from that node down to a leaf.
- **Height of the tree** — the height of the root node.
- **Subtree** — a node and all of its descendants. Every node is the root of its own subtree.

![Tree Anatomy](https://raw.githubusercontent.com/schemabotview/dsa/main/img/tree-anatomy.svg)

## Binary Trees

A **binary tree** is a tree where every node has **at most two children**, conventionally called the **left child** and the **right child**.

Three important shapes:

| Shape | Definition | Example use |
|---|---|---|
| **Full** | Every node has 0 or 2 children (never 1) | Expression trees |
| **Complete** | All levels fully filled except possibly the last, which is filled left-to-right | Heaps |
| **Perfect** | All internal nodes have exactly 2 children and all leaves are at the same depth | Theoretical analysis |
| **Balanced** | Height is O(log n) — no subtree is much taller than the other | AVL trees, Red-Black trees |

A **perfect** binary tree of height h has exactly **2^(h+1) − 1** nodes. This is why balanced trees give you O(log n) operations — the height is logarithmic in the number of nodes.

## Node Implementation

### Python

In [ ]:
from __future__ import annotations
from collections import deque

class TreeNode:
    def __init__(self, val: int,
                 left: TreeNode | None = None,
                 right: TreeNode | None = None):
        self.val   = val
        self.left  = left
        self.right = right

    def __repr__(self) -> str:
        return f"TreeNode({self.val})"

# Build the sample tree used in all examples:
#        4
#       / \
#      2   6
#     / \ / \
#    1  3 5  7

root = TreeNode(4,
    TreeNode(2, TreeNode(1), TreeNode(3)),
    TreeNode(6, TreeNode(5), TreeNode(7))
)

print(root)         # TreeNode(4)
print(root.left)    # TreeNode(2)
print(root.right)   # TreeNode(6)

### Kotlin

```kotlin
data class TreeNode(
    val `val`: Int,
    var left: TreeNode? = null,
    var right: TreeNode? = null
)

// Build the same tree
val root = TreeNode(4,
    TreeNode(2, TreeNode(1), TreeNode(3)),
    TreeNode(6, TreeNode(5), TreeNode(7))
)

println(root.`val`)        // 4
println(root.left?.`val`)  // 2
println(root.right?.`val`) // 6
```

## Tree Traversals

Traversal means visiting every node exactly once in a defined order. There are four standard traversals.

**Depth-first traversals** follow a path deep into the tree before backtracking. All three are naturally recursive:
- **In-order** (Left → Root → Right) — visits nodes in sorted order on a BST
- **Pre-order** (Root → Left → Right) — root first; used to copy or serialise a tree
- **Post-order** (Left → Right → Root) — children first; used to delete a tree or evaluate expressions

**Breadth-first (level-order)** visits nodes level by level using a queue.

![Tree Traversals](https://raw.githubusercontent.com/schemabotview/dsa/main/img/tree-traversals.svg)

### Python — All Four Traversals

In [ ]:
# In-order: Left → Root → Right  →  sorted order on a BST
def inorder(node: TreeNode | None) -> list[int]:
    if node is None:
        return []
    return inorder(node.left) + [node.val] + inorder(node.right)

# Pre-order: Root → Left → Right  →  copy / serialise
def preorder(node: TreeNode | None) -> list[int]:
    if node is None:
        return []
    return [node.val] + preorder(node.left) + preorder(node.right)

# Post-order: Left → Right → Root  →  delete / evaluate
def postorder(node: TreeNode | None) -> list[int]:
    if node is None:
        return []
    return postorder(node.left) + postorder(node.right) + [node.val]

# Level-order (BFS): level by level, left to right
def level_order(root: TreeNode | None) -> list[list[int]]:
    if root is None:
        return []
    result, queue = [], deque([root])
    while queue:
        level_size = len(queue)           # nodes on the current level
        level = []
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
    return result

print("In-order:   ", inorder(root))      # [1, 2, 3, 4, 5, 6, 7]
print("Pre-order:  ", preorder(root))     # [4, 2, 1, 3, 6, 5, 7]
print("Post-order: ", postorder(root))    # [1, 3, 2, 5, 7, 6, 4]
print("Level-order:", level_order(root))  # [[4], [2, 6], [1, 3, 5, 7]]

### Kotlin — All Four Traversals

```kotlin
import java.util.LinkedList

fun inorder(node: TreeNode?): List<Int> {
    if (node == null) return emptyList()
    return inorder(node.left) + node.`val` + inorder(node.right)
}

fun preorder(node: TreeNode?): List<Int> {
    if (node == null) return emptyList()
    return listOf(node.`val`) + preorder(node.left) + preorder(node.right)
}

fun postorder(node: TreeNode?): List<Int> {
    if (node == null) return emptyList()
    return postorder(node.left) + postorder(node.right) + node.`val`
}

fun levelOrder(root: TreeNode?): List<List<Int>> {
    if (root == null) return emptyList()
    val result = mutableListOf<List<Int>>()
    val queue = LinkedList<TreeNode>().also { it.add(root) }
    while (queue.isNotEmpty()) {
        val level = mutableListOf<Int>()
        repeat(queue.size) {
            val node = queue.poll()!!
            level.add(node.`val`)
            node.left?.let  { queue.add(it) }
            node.right?.let { queue.add(it) }
        }
        result.add(level)
    }
    return result
}
```

## Tree Height and Depth

**Height** and **depth** are computed recursively. The height of a node is one plus the maximum of the heights of its children — base case is `None` (height −1) or a leaf (height 0).

### Python

In [ ]:
def height(node: TreeNode | None) -> int:
    """Height of the tree rooted at node. Empty tree → -1."""
    if node is None:
        return -1                              # convention: null height = -1
    return 1 + max(height(node.left), height(node.right))

def depth(root: TreeNode | None, target: int, d: int = 0) -> int:
    """Depth of the node with the given value. Returns -1 if not found."""
    if root is None:
        return -1
    if root.val == target:
        return d
    left  = depth(root.left,  target, d + 1)
    right = depth(root.right, target, d + 1)
    return left if left != -1 else right

print("Tree height:", height(root))          # 2
print("Depth of 1: ", depth(root, 1))        # 2
print("Depth of 6: ", depth(root, 6))        # 1
print("Depth of 4: ", depth(root, 4))        # 0  (root)

### Kotlin

```kotlin
fun height(node: TreeNode?): Int {
    if (node == null) return -1
    return 1 + maxOf(height(node.left), height(node.right))
}

fun depth(root: TreeNode?, target: Int, d: Int = 0): Int {
    if (root == null) return -1
    if (root.`val` == target) return d
    val left  = depth(root.left,  target, d + 1)
    val right = depth(root.right, target, d + 1)
    return if (left != -1) left else right
}
```

## Counting Nodes and Leaves

Both follow the same recursive pattern: solve the subproblem on the left subtree, solve it on the right subtree, combine.

### Python

In [ ]:
def count_nodes(node: TreeNode | None) -> int:
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def count_leaves(node: TreeNode | None) -> int:
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1                              # leaf
    return count_leaves(node.left) + count_leaves(node.right)

print("Total nodes:", count_nodes(root))     # 7
print("Leaf nodes: ", count_leaves(root))    # 4  (1, 3, 5, 7)

### Kotlin

```kotlin
fun countNodes(node: TreeNode?): Int {
    if (node == null) return 0
    return 1 + countNodes(node.left) + countNodes(node.right)
}

fun countLeaves(node: TreeNode?): Int {
    if (node == null) return 0
    if (node.left == null && node.right == null) return 1
    return countLeaves(node.left) + countLeaves(node.right)
}
```

## Checking if a Tree is Balanced

A **height-balanced** binary tree is one where, for every node, the heights of the left and right subtrees differ by at most 1. This is the property that keeps operations at O(log n).

The naive approach checks balance at every node by calling `height()` twice — O(n log n). The efficient approach computes height and balance in a single post-order pass — O(n).

### Python

In [ ]:
def is_balanced(node: TreeNode | None) -> bool:
    """
    O(n) — returns (is_balanced, height) for the subtree.
    Uses a single post-order pass.
    """
    def check(node: TreeNode | None) -> tuple[bool, int]:
        if node is None:
            return True, -1
        left_ok,  lh = check(node.left)
        right_ok, rh = check(node.right)
        balanced = left_ok and right_ok and abs(lh - rh) <= 1
        return balanced, 1 + max(lh, rh)

    return check(node)[0]

# Balanced tree
print(is_balanced(root))    # True

# Unbalanced tree — a chain going left-left-left
skewed = TreeNode(1, TreeNode(2, TreeNode(3, TreeNode(4))))
print(is_balanced(skewed))  # False

### Kotlin

```kotlin
fun isBalanced(node: TreeNode?): Boolean {
    // Returns Pair(isBalanced, height)
    fun check(node: TreeNode?): Pair<Boolean, Int> {
        if (node == null) return Pair(true, -1)
        val (leftOk,  lh) = check(node.left)
        val (rightOk, rh) = check(node.right)
        val balanced = leftOk && rightOk && Math.abs(lh - rh) <= 1
        return Pair(balanced, 1 + maxOf(lh, rh))
    }
    return check(node).first
}
```

## Iterative Traversal with an Explicit Stack

Recursive traversals are clean but risk stack overflow on very deep trees. You can simulate any depth-first traversal iteratively using an explicit stack — the algorithm is identical, you just manage the stack yourself instead of relying on the call stack.

### Python

In [ ]:
def inorder_iterative(root: TreeNode | None) -> list[int]:
    result, stack, node = [], [], root
    while node or stack:
        # Go as far left as possible
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()          # leftmost unvisited node
        result.append(node.val)     # visit
        node = node.right           # move to right subtree
    return result

def preorder_iterative(root: TreeNode | None) -> list[int]:
    if root is None:
        return []
    result, stack = [], [root]
    while stack:
        node = stack.pop()
        result.append(node.val)              # visit root
        if node.right: stack.append(node.right)  # right pushed first (LIFO)
        if node.left:  stack.append(node.left)
    return result

print("Inorder iterative: ", inorder_iterative(root))   # [1,2,3,4,5,6,7]
print("Preorder iterative:", preorder_iterative(root))  # [4,2,1,3,6,5,7]

### Kotlin

```kotlin
fun inorderIterative(root: TreeNode?): List<Int> {
    val result = mutableListOf<Int>()
    val stack = ArrayDeque<TreeNode>()
    var node = root
    while (node != null || stack.isNotEmpty()) {
        while (node != null) { stack.addLast(node); node = node.left }
        node = stack.removeLast()
        result.add(node.`val`)
        node = node.right
    }
    return result
}

fun preorderIterative(root: TreeNode?): List<Int> {
    if (root == null) return emptyList()
    val result = mutableListOf<Int>()
    val stack = ArrayDeque<TreeNode>().also { it.addLast(root) }
    while (stack.isNotEmpty()) {
        val node = stack.removeLast()
        result.add(node.`val`)
        node.right?.let { stack.addLast(it) }
        node.left?.let  { stack.addLast(it) }
    }
    return result
}
```

## Key Takeaways

- A **binary tree** is a hierarchical structure where every node has at most two children. Height is the length of the longest root-to-leaf path.
- A **balanced** tree has height O(log n), enabling O(log n) search, insert, and delete. An unbalanced tree degrades to O(n) in the worst case (a linked list).
- There are four standard traversals: **in-order** (sorted order on BSTs), **pre-order** (root first; serialisation), **post-order** (children first; deletion/evaluation), and **level-order** (BFS; shortest path, level-by-level).
- All depth-first traversals follow the same recursive skeleton: handle `None` as the base case, recurse into the left and right subtrees, combine results. The only difference is when you visit the current node.
- **Level-order** uses a queue (deque), not a stack — enqueue children left-to-right and process one level at a time.
- Recursive traversals are O(n) time and O(h) space where h is the tree height. For a balanced tree that is O(log n) space; for a skewed tree it is O(n).
- You can replace any recursive traversal with an iterative one using an explicit stack — same logic, no call-stack depth limit.
- The balanced-tree check runs in O(n) by combining height computation and balance verification in a single post-order pass — avoid the naive O(n log n) approach.